In [ ]:
# import system libs
import os
import random
import shutil


Data repository  (400x)
Cancer is sorted first, then normal tissues

In [ ]:
# Source and directories for CANCER at 400x
source1 = '/content/drive/MyDrive/DATA8009 OSCC Dataets/ftmp4cvtmb-1/Second Set/400x OSCC Histopathological Images'
train_dir1 = '/content/drive/MyDrive/DATA8009 OSCC Dataets/Sorted oscc/Train/Cancer'
val_dir1 = '/content/drive/MyDrive/DATA8009 OSCC Dataets/Sorted oscc/Val/Cancer'
test_dir1 = '/content/drive/MyDrive/DATA8009 OSCC Dataets/Sorted oscc/Test/Cancer'

In [ ]:
def split_data(source, train_dir, val_dir, test_dir, split_ratio=(0.7, 0.2, 0.1)):
    # 1. Get list of all files
    files = os.listdir(source)
    random.shuffle(files)

    # 2. Calculate split indices
    train_count = int(len(files) * split_ratio[0])
    val_count = int(len(files) * split_ratio[1])

    # 3. Slice the list
    train_files = files[:train_count]
    val_files = files[train_count:train_count + val_count]
    test_files = files[train_count + val_count:]

    # 4. Move files
    for f in train_files:
        shutil.copy(os.path.join(source, f), os.path.join(train_dir, f))
    for f in val_files:
        shutil.copy(os.path.join(source, f), os.path.join(val_dir, f))
    for f in test_files:
        shutil.copy(os.path.join(source, f), os.path.join(test_dir, f))

In [ ]:
split_data(source1, train_dir1, val_dir1, test_dir1)

In [ ]:
# Source and directories for NORMAL at 400x
source2 = '/content/drive/MyDrive/DATA8009 OSCC Dataets/ftmp4cvtmb-1/Second Set/400x Normal Oral Cavity Histopathological Images'
train_dir2 = '/content/drive/MyDrive/DATA8009 OSCC Dataets/Sorted oscc/Train/Normal'
val_dir2 = '/content/drive/MyDrive/DATA8009 OSCC Dataets/Sorted oscc/Val/Normal'
test_dir2 = '/content/drive/MyDrive/DATA8009 OSCC Dataets/Sorted oscc/Test/Normal'

In [ ]:
split_data(source2, train_dir2, val_dir2, test_dir2)

Preprocessing Stage

Simple CNN for classification (ResNet50)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam

# Data generators with augmentation (only for training)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.1
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    '/content/drive/MyDrive/DATA8009 OSCC Dataets/Sorted oscc/Train',
    target_size=(224, 224),
    batch_size=16,
    class_mode='binary'
)

val_generator = val_test_datagen.flow_from_directory(
    '/content/drive/MyDrive/DATA8009 OSCC Dataets/Sorted oscc/Val',
    target_size=(224, 224),
    batch_size=16,
    class_mode='binary'
)

# Transfer learning
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False   # freeze base first

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer=Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])

# Train
history = model.fit(train_generator, validation_data=val_generator, epochs=20)

# Unfreeze some top layers for fine-tuning if needed later

Found 634 images belonging to 2 classes.
Found 250 images belonging to 2 classes.
Epoch 1/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 79s 2s/step - accuracy: 0.6325 - loss: 0.7007 - val_accuracy: 0.7200 - val_loss: 0.5924
Epoch 2/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 57s 1s/step - accuracy: 0.6830 - loss: 0.6316 - val_accuracy: 0.7200 - val_loss: 0.5902
Epoch 3/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 58s 1s/step - accuracy: 0.7019 - loss: 0.6462 - val_accuracy: 0.7200 - val_loss: 0.5888
Epoch 4/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 58s 1s/step - accuracy: 0.7003 - loss: 0.6441 - val_accuracy: 0.7200 - val_loss: 0.5890
Epoch 5/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 58s 1s/step - accuracy: 0.6940 - loss: 0.6225 - val_accuracy: 0.7200 - val_loss: 0.5944
Epoch 6/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 58s 1s/step - accuracy: 0.7066 - loss: 0.6289 - val_accuracy: 0.7200 - val_loss: 0.5954
Epoch 7/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 58s 1s/step - accuracy: 0.7145 - loss: 0.6228 - val_accuracy: 0.7200 - val_loss: 0.5890
Epoch 8/20
40/40 ━━━━━━━━━━━━━━━━━━━

Ignore next two cells

In [ ]:
import os

train_path = '/content/drive/MyDrive/DATA8009 OSCC Dataets/Sorted oscc/Train'
print("All items in train folder:", os.listdir(train_path))

# Check if any are files (should only be folders)
for item in os.listdir(train_path):
    item_path = os.path.join(train_path, item)
    if os.path.isfile(item_path):
        print(f"Found file (not folder): {item}")
    elif os.path.isdir(item_path):
        files_in_folder = len(os.listdir(item_path))
        print(f"Folder: {item} - contains {files_in_folder} images")

All items in train folder: ['Normal', 'Cancer']
Folder: Normal - contains 178 images
Folder: Cancer - contains 456 images


In [ ]:
import os
import shutil

# Remove .ipynb_checkpoints from train, val, and test folders
for split in ['Train', 'Val', 'Test']:
    checkpoint_path = f'/content/drive/MyDrive/DATA8009 OSCC Dataets/Sorted oscc/{split}/.ipynb_checkpoints'
    if os.path.exists(checkpoint_path):
        shutil.rmtree(checkpoint_path)
        print(f"Removed from {split}")

Save model


In [ ]:
# Save for later use
model.save('histopathology_cnn_model.h5')

Evaluation

In [ ]:
# Load test data
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    '/content/drive/MyDrive/DATA8009 OSCC Dataets/Sorted oscc/Test',
    target_size=(224, 224),
    batch_size=16,
    class_mode='binary',
    shuffle=False  # Important for evaluation
)

Found 136 images belonging to 2 classes.


In [ ]:
# Get loss and accuracy
test_loss, test_acc = model.evaluate(test_generator)
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")

9/9 ━━━━━━━━━━━━━━━━━━━━ 30s 4s/step - accuracy: 0.6985 - loss: 0.6177
Test accuracy: 0.6985
Test loss: 0.6177


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Predict probabilities
predictions = model.predict(test_generator)
predicted_classes = (predictions > 0.5).astype(int)

# Get true labels
true_classes = test_generator.classes

# Classification report
print(classification_report(true_classes, predicted_classes,
                          target_names=['normal', 'cancerous']))

# Confusion matrix
print(confusion_matrix(true_classes, predicted_classes))

# AUC-ROC score
auc_score = roc_auc_score(true_classes, predictions)
print(f"AUC-ROC: {auc_score:.4f}")

9/9 ━━━━━━━━━━━━━━━━━━━━ 19s 1s/step
              precision    recall  f1-score   support

      normal       0.70      1.00      0.82        95
   cancerous       0.00      0.00      0.00        41

    accuracy                           0.70       136
   macro avg       0.35      0.50      0.41       136
weighted avg       0.49      0.70      0.57       136

[[95  0]
 [41  0]]
AUC-ROC: 0.4467


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
